import библиотек

In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.datasets import mnist

Скачиваем датасет

In [2]:
from tensorflow.keras.datasets import mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test = X_test.reshape(-1, 784).astype('float32') / 255.0

# Переводим на onehot

def onehot(y, num_classes = 10):
  one_hot = np.zeros((y.size, num_classes))
  one_hot[np.arange(y.size), y]= 1.0
  return one_hot

y_train_one_hot = onehot(y_train)
y_test_one_hot = onehot(y_test)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Создание слоев

In [3]:
layer_size = [784, 128, 64, 10]
weight_list = []
bias_list = []
for i in range(len(layer_size) - 1):
  coff = np.sqrt(2.0 / layer_size[i])
  weights = np.random.randn(layer_size[i], layer_size[i + 1]) * coff
  bias = np.zeros((1, layer_size[i + 1]))
  bias_list.append(bias)
  weight_list.append(weights)

Создание функций для обучения

In [4]:
def Relu(x):
  return np.maximum(x, 0)
def Softmax(x):
    max_x = np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x - max_x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

Цикл прохода Forward Pass

In [5]:
def ForwardPass(x, weight_list, bias_list):
  activations = [x]
  current_signal = x
  num_layers = len(weight_list)
  for i in range(num_layers):
    z = np.dot(current_signal, weight_list[i]) + bias_list[i]
    if i == num_layers - 1:
      current_signal = Softmax(z)
    else:
      current_signal = Relu(z)
    activations.append(current_signal)
  return activations

Loss Function

In [6]:
def LossFunction(predictions, y_true):
  predictions = np.clip(predictions, 1e-15, 1.0 - 1e-15)
  return -np.sum(y_true * np.log(predictions)) / y_true.shape[0]

BackPropagation

In [7]:
def derivative(z):
  return (z > 0).astype(float)
def BackPropagation(activations, weight_list, y_true):
  db = []
  dw = []
  error = activations[-1] - y_true

  # Складываем в начало списков (так как идем с конца)
  dw.insert(0, dw)
  db.insert(0, db)

        # Если мы еще не дошли до входа, передаем ошибку на слой глубже
  if i > 0:
            # Ошибка для скрытого слоя зависит от весов и производной ReLU
    error = np.dot(error, weight_list[i].T) * derivative(activations[i])

  return dw, db

Цикл обучения

In [10]:
learning_rate = 0.1
batch_size = 64
epochs = 50
for epoch in range(epochs):
  permutation = np.random.permutation(X_train.shape[0])
  x_train_shuffled = X_train[permutation]
  y_train_shuffled = y_train_one_hot[permutation]
  for step in range(0, X_train.shape[0], batch_size):
    x_batch = x_train_shuffled[step:step + batch_size]
    y_batch = y_train_shuffled[step:step + batch_size]
    activations = ForwardPass(x_batch, weight_list, bias_list)
    dw_list = []
    db_list = []
    # Ошибка на выходе (Предсказание сети - Правильный ответ)
    error = activations[-1] - y_batch
    for j in reversed(range(len(weight_list))):
      # Считаем градиенты для текущего слоя (усредняем по размеру батча)
      dw = np.dot(activations[j].T, error) / x_batch.shape[0]
      db = np.sum(error, axis=0, keepdims=True) / x_batch.shape[0]

      dw_list.insert(0, dw)
      db_list.insert(0, db)

      if j > 0:
        error = np.dot(error, weight_list[j].T) * derivative(activations[j])

    for j in range(len(weight_list)):
      weight_list[j] -= learning_rate * dw_list[j]
      bias_list[j] -= learning_rate * db_list[j]
  test_activations = ForwardPass(X_test, weight_list, bias_list)
  predictions = np.argmax(test_activations[-1], axis=1)
  accuracy = np.mean(predictions == y_test) * 100

  print(f"Эпоха {epoch + 1}/{epochs} Завершена. Точность на тесте: {accuracy:.2f}%")

print("Обучение завершено!")

Эпоха 1/50 Завершена. Точность на тесте: 98.20%
Эпоха 2/50 Завершена. Точность на тесте: 98.16%
Эпоха 3/50 Завершена. Точность на тесте: 98.21%
Эпоха 4/50 Завершена. Точность на тесте: 98.13%
Эпоха 5/50 Завершена. Точность на тесте: 98.19%
Эпоха 6/50 Завершена. Точность на тесте: 98.17%
Эпоха 7/50 Завершена. Точность на тесте: 98.22%
Эпоха 8/50 Завершена. Точность на тесте: 98.16%
Эпоха 9/50 Завершена. Точность на тесте: 98.19%
Эпоха 10/50 Завершена. Точность на тесте: 98.13%
Эпоха 11/50 Завершена. Точность на тесте: 98.22%
Эпоха 12/50 Завершена. Точность на тесте: 98.21%
Эпоха 13/50 Завершена. Точность на тесте: 98.22%
Эпоха 14/50 Завершена. Точность на тесте: 98.18%
Эпоха 15/50 Завершена. Точность на тесте: 98.20%
Эпоха 16/50 Завершена. Точность на тесте: 98.22%
Эпоха 17/50 Завершена. Точность на тесте: 98.20%
Эпоха 18/50 Завершена. Точность на тесте: 98.21%
Эпоха 19/50 Завершена. Точность на тесте: 98.21%
Эпоха 20/50 Завершена. Точность на тесте: 98.19%
Эпоха 21/50 Завершена. Точнос